<a href="https://colab.research.google.com/github/vik000/AdvancedNode/blob/master/notebooks/G0_E4_joint_1_5B_after_E3a.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================================
#  G0 · E4 — JOINT MULTI-TASK ADAPTER.  ONE cell.
#  Notebook name: G0_E4_joint_1.5B_after_E3a.ipynb
#
#  THE QUESTION
#    Gate 0c failed: 75.6% of the adapters' effect is explained by WHICH
#    QUESTION was asked rather than which adapter was active, and the eight
#    adapters have row-to-row correlation 0.898. Suspected cause: each
#    adapter is trained on ONE question and therefore never learns that the
#    question is something to condition on.
#
#    E4 trains ONE LoRA on all eight questions mixed together. Telling the
#    dimensions apart becomes the training objective instead of an accident.
#
#  ONE VARIABLE ONLY
#    Same loss as E1 (plain cross-entropy on the answer digit). Not soft
#    targets, not expectation regression — E3a established that the
#    calibration slope is not a training problem, so it is now handled at
#    readout. Architecture is the only thing that changes here.
#
#  ACCEPTANCE TEST — 0e, not 0c
#    With a single adapter there is no leakage matrix to build. The question
#    becomes whether the readings are more separable:
#        effective dims   3.70 (E1, eight adapters)  ->  > 3.70, toward 4.79
#        mean r           must hold, not collapse
#
#  ~90 min. Prints a progress check at 25 / 50 / 75% so you can stop early.
#  Saves to MyDrive/affective-layer/adapters/<model>__joint/
# =====================================================================
MODEL_ID  = "Qwen/Qwen2.5-1.5B-Instruct"
PER_DIM   = 2000        # 2000 x 8 dims = 16000 examples (E1 used 4000 x 8 across 8 runs)
EPOCHS    = 2
LORA_R    = 16          # doubled: one adapter now carries all eight dimensions

# ---------------------------------------------------------------- setup
%pip -q install -U transformers peft bitsandbytes accelerate datasets

import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
import torch, gc, json, io, zipfile, random, inspect
import numpy as np, pandas as pd, requests as rq
from google.colab import drive
assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"
drive.mount('/content/drive')

SEED, MAX_LEN, LR = 0, 256, 1e-4
BATCH, GRAD_ACC, LORA_ALPHA = 4, 4, 32
DIAG_N, DIAG_SEED, PROG_N = 300, 123, 80

ROOT = "/content/drive/MyDrive/affective-layer"
OUT  = f"{ROOT}/adapters/{MODEL_ID.split('/')[-1]}__joint"
RES  = f"{ROOT}/results"
os.makedirs(OUT, exist_ok=True); os.makedirs(RES, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
TAG = f"G0_E4_joint_{MODEL_ID.split('/')[-1]}"
print("run tag:", TAG, "| adapter ->", OUT)

DIMENSIONS = {
 "valence":     dict(src="emobank", q="How positive versus negative is the emotional state expressed by the writer? (1 = extremely negative, 9 = extremely positive)"),
 "arousal":     dict(src="emobank", q="How activated or agitated versus calm is the emotional state expressed by the writer? (1 = completely calm, 9 = extremely agitated)"),
 "relevance":   dict(src="envent",  q="How relevant does the writer consider this event to their own goals and needs? (1 = not at all relevant, 9 = extremely relevant)"),
 "novelty":     dict(src="envent",  q="How sudden or unexpected was this event for the writer? (1 = fully expected, 9 = completely sudden and unexpected)"),
 "conduciveness": dict(src="envent", q="To what degree does the writer appraise this event as helping rather than obstructing their goals? (1 = severely obstructs, 9 = strongly helps)"),
 "agency":      dict(src="envent",  q="To what degree does the writer appraise this event as caused by another person or group, rather than by themselves or circumstances? (1 = not at all other-caused, 9 = entirely other-caused)"),
 "coping":      dict(src="envent",  q="How much does the writer appraise the situation as one they can influence or control? (1 = no control at all, 9 = full control)"),
 "norm_compatibility": dict(src="envent", q="To what degree does the writer appraise this event as compatible with social and moral standards? (1 = severe violation of standards, 9 = fully compatible)"),
}
TEXT_COL = "generated_text"
APPRAISAL_MAP = {"relevance":["goal_relevance"], "novelty":["suddenness"],
                 "conduciveness":["goal_support"], "agency":["other_responsblt"],
                 "coping":["self_control"], "norm_compatibility":["standards","social_norms"]}
FLIP = {"norm_compatibility"}
APPR = [d for d,s in DIMENSIONS.items() if s["src"]=="envent"]

# ----------------------------------------------------------------- data
eb = pd.read_csv("https://raw.githubusercontent.com/JULIELab/EmoBank/master/corpus/emobank.csv").rename(columns=str.strip)
zf = zipfile.ZipFile(io.BytesIO(rq.get("https://www.romanklinger.de/data-sets/crowd-enVent2023.zip", timeout=300).content))
cand = [n for n in zf.namelist() if n.endswith((".tsv",".csv")) and "gener" in n.lower()]
ev = pd.read_csv(zf.open(cand[0]), sep="\t" if cand[0].endswith(".tsv") else ",")
def rescale(x, lo, hi): return 1 + (x - lo) * 8.0 / (hi - lo)

datasets, pool = {}, []
for d, spec in DIMENSIONS.items():
    if spec["src"] == "emobank":
        col = "V" if d == "valence" else "A"
        df = eb[["text", col, "split"]].dropna().rename(columns={col:"y"})
        df["rating"] = rescale(df.y, 1, 5)
        tr, va = df[df.split=="train"], df[df.split!="train"]
    else:
        cols = APPRAISAL_MAP[d]
        df = ev[[TEXT_COL]+cols].dropna().rename(columns={TEXT_COL:"text"})
        df["y"] = df[cols].mean(axis=1)
        df["rating"] = rescale(df.y, df.y.min(), df.y.max())
        if d in FLIP: df["rating"] = 10 - df["rating"]
        va = df.sample(frac=0.15, random_state=SEED); tr = df.drop(va.index)
    tr = tr.sample(min(len(tr), PER_DIM), random_state=SEED)
    datasets[d] = (tr.reset_index(drop=True), va.reset_index(drop=True))
    t = tr[["text","rating"]].copy(); t["dim"] = d
    pool.append(t)
    print(f"  {d:<20} train={len(tr):5d} val={len(va):5d}")

MIX = pd.concat(pool).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
print(f"\njoint training pool: {len(MIX)} examples across {MIX.dim.nunique()} dimensions")
print(f"steps: {int(len(MIX)*EPOCHS/(BATCH*GRAD_ACC))}")

# ---------------------------------------------------------------- model
import transformers
from transformers import (AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
                          Trainer, TrainingArguments, TrainerCallback)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training, PeftModel
print("transformers", transformers.__version__)

_TA_OK = set(inspect.signature(TrainingArguments.__init__).parameters)
def make_args(**kw):
    drop = [k for k in kw if k not in _TA_OK]
    if drop: print("  [note] ignored:", drop)
    return TrainingArguments(**{k:v for k,v in kw.items() if k in _TA_OK})

tok = AutoTokenizer.from_pretrained(MODEL_ID)
if tok.pad_token is None: tok.pad_token = tok.eos_token
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
def load_base():
    m = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb,
                                             device_map="auto", torch_dtype=torch.float16)
    m.config.use_cache = False
    return m

SYS = "You are a careful annotator who rates texts on psychological appraisal dimensions."
def make_prompt(text, question):
    msgs = [{"role":"system","content":SYS},
            {"role":"user","content":f'Text: "{text}"\n\n{question}\nAnswer with a single integer from 1 to 9.'}]
    return tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
DIGITS = [tok.encode(str(i), add_special_tokens=False)[0] for i in range(1,10)]
assert len(set(DIGITS)) == 9

# --------------------------------------------- dataset (mixed questions)
class JointDS(torch.utils.data.Dataset):
    def __init__(self, df): self.rows = df.to_dict("records")
    def __len__(self): return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        q = DIMENSIONS[r["dim"]]["q"]
        pid = tok(make_prompt(str(r["text"])[:1200], q),
                  truncation=True, max_length=MAX_LEN)["input_ids"]
        ans = DIGITS[int(np.clip(round(r["rating"]),1,9))-1]
        return dict(input_ids=pid+[ans, tok.eos_token_id],
                    labels=[-100]*len(pid)+[ans, tok.eos_token_id])

def collate(b):
    L, pad = max(len(x["input_ids"]) for x in b), tok.pad_token_id
    return dict(
        input_ids=torch.tensor([x["input_ids"]+[pad]*(L-len(x["input_ids"])) for x in b]),
        labels=torch.tensor([x["labels"]+[-100]*(L-len(x["labels"])) for x in b]),
        attention_mask=torch.tensor([[1]*len(x["input_ids"])+[0]*(L-len(x["input_ids"])) for x in b]))

@torch.no_grad()
def rate(model, texts, question, bs=16):
    prev, tok.padding_side = tok.padding_side, "left"
    out = []
    for i in range(0, len(texts), bs):
        prompts = [make_prompt(str(t)[:1200], question) for t in texts[i:i+bs]]
        enc = tok(prompts, return_tensors="pt", padding=True,
                  truncation=True, max_length=MAX_LEN).to(model.device)
        p = torch.softmax(model(**enc).logits[:, -1, :][:, DIGITS].float(), -1)
        out += (p @ torch.arange(1,10,dtype=torch.float,device=p.device)).tolist()
    tok.padding_side = prev
    return np.array(out)

def effective_dims(C):
    e = np.linalg.eigvalsh(C.values.astype(float))[::-1]
    return float(e.sum()**2/(e**2).sum())

# ------------------------------------------- held-out set for 0e / progress
need = sorted({c for d in APPR for c in APPRAISAL_MAP[d]})
com = ev.dropna(subset=[TEXT_COL]+need).copy()
H = pd.DataFrame(index=com.index)
for d in APPR:
    cols = APPRAISAL_MAP[d]; full = ev[cols].dropna().mean(axis=1)
    rr = rescale(com[cols].mean(axis=1), full.min(), full.max())
    H[d] = (10-rr) if d in FLIP else rr
seen = set()
for d in DIMENSIONS: seen |= set(datasets[d][0].text.astype(str))
keep = ~com[TEXT_COL].astype(str).isin(seen)
com, H = com[keep], H[keep]
idx  = com.sample(min(DIAG_N,len(com)), random_state=DIAG_SEED).index
TEXTS = com.loc[idx, TEXT_COL].astype(str).tolist(); H = H.loc[idx].reset_index(drop=True)
PIDX  = list(range(min(PROG_N, len(TEXTS))))
print(f"held-out pool {len(com)} | diagnostic {len(TEXTS)} | progress checks on {len(PIDX)}")

class Progress(TrainerCallback):
    def __init__(self, model_ref): self.m = model_ref; self.done = set()
    def on_step_end(self, args, state, control, **kw):
        tot = state.max_steps
        for frac in (.25,.50,.75):
            mark = int(tot*frac)
            if state.global_step == mark and frac not in self.done:
                self.done.add(frac)
                self.m.eval()
                sub = [TEXTS[i] for i in PIDX]
                R = {d: rate(self.m, sub, DIMENSIONS[d]["q"]) for d in APPR}
                R = pd.DataFrame(R)
                rs = [np.corrcoef(R[d], H[d].values[PIDX])[0,1] for d in APPR]
                print(f"\n  --- {int(frac*100)}% --- mean r {np.mean(rs):.3f} | "
                      f"effective dims {effective_dims(R.corr()):.2f}   "
                      f"(E1: r 0.646, dims 3.70 | humans 4.79)\n", flush=True)
                self.m.train()

# ---------------------------------------------------------------- train
if os.path.exists(f"{OUT}/adapter_model.safetensors"):
    print("[skip] joint adapter already trained — loading")
    base = load_base(); model = PeftModel.from_pretrained(base, OUT, adapter_name="joint")
else:
    base = load_base()
    model = get_peft_model(prepare_model_for_kbit_training(base),
        LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, bias="none",
                   task_type="CAUSAL_LM",
                   target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]))
    model.print_trainable_parameters()
    args = make_args(
        output_dir="/content/tmp_joint", num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH, gradient_accumulation_steps=GRAD_ACC,
        learning_rate=LR, fp16=True, logging_steps=50, save_strategy="no",
        optim="paged_adamw_8bit", report_to="none", seed=SEED,
        gradient_checkpointing=True, lr_scheduler_type="cosine",
        remove_unused_columns=False)
    tr = Trainer(model=model, args=args, train_dataset=JointDS(MIX), data_collator=collate)
    tr.add_callback(Progress(model))
    tr.train()
    model.save_pretrained(OUT)
    print("saved ->", OUT)
    del tr; gc.collect(); torch.cuda.empty_cache()

# ------------------------------------------------------------- diagnose
model.eval()
Ra = pd.DataFrame({d: rate(model, TEXTS, DIMENSIONS[d]["q"]) for d in APPR})
print("adapted ratings done")
del model; gc.collect(); torch.cuda.empty_cache(); torch.cuda.ipc_collect()
base2 = load_base()
Rb = pd.DataFrame({d: rate(base2, TEXTS, DIMENSIONS[d]["q"]) for d in APPR})
print("base ratings done")

rows=[]
for d in APPR:
    h = H[d].values
    for cond,R in (("base",Rb),("adapted",Ra)):
        x = R[d].values; sl, ic = [float(z) for z in np.polyfit(h,x,1)]
        rows.append(dict(dim=d, cond=cond, r=float(np.corrcoef(x,h)[0,1]),
                         slope=sl, intercept=ic, lam=float(x.std()/h.std())))
G = pd.DataFrame(rows); piv = G.pivot(index="dim", columns="cond")
delta = (piv[("r","adapted")]-piv[("r","base")]).rename("delta_r")

print("\n=== 0a / 0d · fidelity + base control ===")
print(pd.concat([piv[("r","base")].rename("r_base"),
                 piv[("r","adapted")].rename("r_adapted"), delta], axis=1).round(3))
mean_r = float(piv[("r","adapted")].mean())
print(f"mean r = {mean_r:.3f}   (E1 eight adapters, 7B: 0.646)")

print("\n=== 0a' · calibration (reported, NOT a target — slope = r x lambda) ===")
print(G[G.cond=="adapted"][["dim","r","slope","intercept","lam"]].set_index("dim").round(3))

Ch, Cb, Ca = H[APPR].corr(), Rb[APPR].corr(), Ra[APPR].corr()
ed_h, ed_b, ed_a = effective_dims(Ch), effective_dims(Cb), effective_dims(Ca)
off = lambda C: float(np.abs(C.values[~np.eye(len(C),dtype=bool)]).mean())

print("\n" + "="*64)
print("  E4 ACCEPTANCE TEST · 0e separability")
print("="*64)
print(f"  humans                     effective dims {ed_h:.2f}   mean|r| {off(Ch):.3f}")
print(f"  frozen base                effective dims {ed_b:.2f}   mean|r| {off(Cb):.3f}")
print(f"  >>> JOINT adapter (E4)     effective dims {ed_a:.2f}   mean|r| {off(Ca):.3f}")
print(f"      E1, eight adapters                    3.70")
print("="*64)
if ed_a > 3.90 and mean_r >= 0.58:
    print("  VERDICT: JOINT WINS. Separability improved. Promote to the 7B.")
elif ed_a > 3.70 and mean_r >= 0.58:
    print("  VERDICT: MARGINAL improvement. Send me the numbers.")
elif mean_r < 0.58:
    print("  VERDICT: fidelity dropped — joint training cost accuracy. Send me the numbers.")
else:
    print("  VERDICT: NO improvement. Per-dimension decomposition is not the cause of 0c.")
print("="*64)
print("\nadapted correlation matrix:\n", Ca.round(2).to_string())

G.to_csv(f"{RES}/calibration__{TAG}.csv", index=False)
for nm,C in (("human",Ch),("base",Cb),("adapted",Ca)): C.to_csv(f"{RES}/corr_{nm}__{TAG}.csv")
json.dump(dict(tag=TAG, model=MODEL_ID, arch="joint_multitask", lora_r=LORA_R,
               per_dim=PER_DIM, epochs=EPOCHS, n=len(TEXTS), mean_r=mean_r,
               eff_dims=dict(human=ed_h, base=ed_b, adapted=ed_a),
               e1_eff_dims=3.70, delta_r={k:float(v) for k,v in delta.items()}),
          open(f"{RES}/summary__{TAG}.json","w"), indent=2)

import shutil
from google.colab import files
stage=f"/content/{TAG}"; os.makedirs(stage, exist_ok=True)
for f in os.listdir(RES):
    if TAG in f: shutil.copy(f"{RES}/{f}", stage)
zp = shutil.make_archive(f"/content/{TAG}", "zip", stage)
print("\nzip:", zp)
files.download(zp)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 18.8 MB/s eta 0:00:00
Mounted at /content/drive
run tag: G0_E4_joint_Qwen2.5-1.5B-Instruct | adapter -> /content/drive/MyDrive/affective-layer/adapters/Qwen2.5-1.5B-Instruct__joint
  valence              train= 2000 val= 1999
  arousal              train= 2000 val= 1999
  relevance            train= 2000 val=  990
  novelty              train= 2000 val=  990
  conduciveness        train= 2000 val=  990
  agency               train= 2000 val=  990
  coping               train= 2000 val=  990
  norm_compatibility   train= 2000 val=  990

joint training pool: 16000 examples across 8 dimensions
steps: 2000
transformers 5.17.0


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...


  --- 25% --- mean r 0.620 | effective dims 2.72   (E1: r 0.646, dims 3.70 | humans 4.79)


  --- 50% --- mean r 0.645 | effective dims 2.81   (E1: r 0.646, dims 3.70 | humans 4.79)


  --- 75% --- mean r 0.653 | effective dims 3.16   (E1: r 0.646, dims 3.70 | humans 4.79)

saved -> /content/drive/MyDrive/affective-layer/adapters/Qwen2.5-1.5B-Instruct__joint
adapted ratings done


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

base ratings done

=== 0a / 0d · fidelity + base control ===
                    r_base  r_adapted  delta_r
dim                                           
agency               0.233      0.666    0.433
conduciveness        0.327      0.623    0.296
coping               0.091      0.490    0.399
norm_compatibility   0.278      0.674    0.395
novelty              0.289      0.581    0.292
relevance            0.002      0.524    0.522
mean r = 0.593   (E1 eight adapters, 7B: 0.646)

=== 0a' · calibration (reported, NOT a target — slope = r x lambda) ===
                        r  slope  intercept    lam
dim                                               
relevance           0.524  0.281      3.761  0.536
novelty             0.581  0.430      3.094  0.741
conduciveness       0.623  0.450      2.287  0.722
agency              0.666  0.450      3.066  0.675
coping              0.490  0.289      3.031  0.591
norm_compatibility  0.674  0.494      3.535  0.733

  E4 ACCEPTANCE TEST · 0e separab

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>